
# HG-BPM: End-to-End Experimental Pipeline

This notebook provides a complete, reproducible implementation of the HG-BPM framework
used to generate the Results and Discussion (Tables 1–8) reported in the manuscript
submitted to *PeerJ Computer Science* as an **AI Application**.

The notebook covers:
- Dataset preparation and preprocessing
- Training/testing split (original and augmented data)
- CTGAN-based data augmentation
- Extra Trees feature selection
- GA-optimized XGBoost and LightGBM models
- Comparative evaluation and ablation studies


## 1. Import Required Libraries

In [ ]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression

import xgboost as xgb
import lightgbm as lgb

from ctgan import CTGAN


## 2. Load Dataset (Table 1: Feature Description)

In [ ]:

# Update the dataset path if required
data = pd.read_csv("data/dhh_students.csv")

target = "Performance_Class"
X = data.drop(columns=[target])
y = data[target]


## 3. Encode Categorical Features

In [ ]:

for col in X.columns:
    if X[col].dtype == "object":
        X[col] = LabelEncoder().fit_transform(X[col])


## 4. Train–Test Split (Table 2: Original Dataset)

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Original Training Samples:", X_train.shape[0])
print("Original Testing Samples:", X_test.shape[0])


## 5. Feature Scaling

In [ ]:

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


## 6. Data Augmentation using CTGAN (Table 2: Augmented Dataset)

In [ ]:

ctgan = CTGAN(epochs=300, verbose=False)
ctgan.fit(pd.concat([X_train, y_train], axis=1))

synthetic_data = ctgan.sample(837)

X_syn = synthetic_data.drop(columns=[target])
y_syn = synthetic_data[target]

X_train_aug = np.vstack((X_train_s, scaler.transform(X_syn)))
y_train_aug = np.concatenate((y_train, y_syn))

print("Augmented Training Samples:", X_train_aug.shape[0])


## 7. Feature Selection using Extra Trees (Table 4)

In [ ]:

xt = ExtraTreesClassifier(n_estimators=300, random_state=42)
xt.fit(X_train_aug, y_train_aug)

importances = xt.feature_importances_
top_indices = np.argsort(importances)[::-1][:10]

selected_features = X.columns[top_indices]
print("Top 10 Selected Features:")
for f in selected_features:
    print(f)

X_train_sel = X_train_aug[:, top_indices]
X_test_sel = X_test_s[:, top_indices]


## 8. GA-Optimized XGBoost Model (Table 3)

In [ ]:

xgb_model = xgb.XGBClassifier(
    n_estimators=320,
    learning_rate=0.08,
    max_depth=7,
    subsample=0.82,
    colsample_bytree=0.76,
    gamma=1.4,
    eval_metric="mlogloss",
    random_state=42
)


## 9. GA-Optimized LightGBM Model (Table 3)

In [ ]:

lgb_model = lgb.LGBMClassifier(
    n_estimators=290,
    learning_rate=0.09,
    max_depth=9,
    num_leaves=96,
    feature_fraction=0.79,
    bagging_fraction=0.87,
    random_state=42
)


## 10. HG-BPM Stacked Ensemble Model

In [ ]:

xgb_model.fit(X_train_sel, y_train_aug)
lgb_model.fit(X_train_sel, y_train_aug)

xgb_train = xgb_model.predict_proba(X_train_sel)
lgb_train = lgb_model.predict_proba(X_train_sel)

stacked_train = np.hstack((xgb_train, lgb_train))

meta_model = LogisticRegression(max_iter=500)
meta_model.fit(stacked_train, y_train_aug)


## 11. Evaluation of HG-BPM (Tables 5, 7, and 8)

In [ ]:

xgb_test = xgb_model.predict_proba(X_test_sel)
lgb_test = lgb_model.predict_proba(X_test_sel)

stacked_test = np.hstack((xgb_test, lgb_test))
final_predictions = meta_model.predict(stacked_test)

print("Accuracy:", accuracy_score(y_test, final_predictions) * 100)
print("Precision:", precision_score(y_test, final_predictions, average='weighted') * 100)
print("Recall:", recall_score(y_test, final_predictions, average='weighted') * 100)
print("F1-Score:", f1_score(y_test, final_predictions, average='weighted') * 100)


## 12. Comparative Evaluation with Baseline Models (Table 5)

In [ ]:

baseline_models = {
    "DT": DecisionTreeClassifier(),
    "SVM": SVC(probability=True),
    "KNN": KNeighborsClassifier(),
    "NB": GaussianNB(),
    "RF": RandomForestClassifier(),
    "XT": ExtraTreesClassifier(n_estimators=300),
}

for name, model in baseline_models.items():
    model.fit(X_train_sel, y_train_aug)
    preds = model.predict(X_test_sel)
    print(name,
          accuracy_score(y_test, preds) * 100,
          f1_score(y_test, preds, average='weighted') * 100)


## 13. Ablation Study (Tables 7 and 8)

In [ ]:

def evaluate_model(model, Xtr, ytr, Xte, yte):
    model.fit(Xtr, ytr)
    p = model.predict(Xte)
    return (
        accuracy_score(yte, p) * 100,
        precision_score(yte, p, average='weighted') * 100,
        recall_score(yte, p, average='weighted') * 100,
        f1_score(yte, p, average='weighted') * 100
    )

print("XGB:", evaluate_model(xgb_model, X_train_sel, y_train_aug, X_test_sel, y_test))
print("LightGBM:", evaluate_model(lgb_model, X_train_sel, y_train_aug, X_test_sel, y_test))
print("ET + XGB + LightGBM + GA (HG-BPM):",
      evaluate_model(meta_model, stacked_train, y_train_aug, stacked_test, y_test))
